# 02_descriptives  (oesophagus)

Reads only data_derived\cohort_clean.csv, the single source of truth built by 01_preprocessing, and describes the cohort: the predictors, their prevalence and missingness, the candidate outcomes, a Table 1 split by the two headline outcomes, and a univariable odds ratio screen across all outcomes. Everything is written into one Excel workbook with an information sheet.

**Decision point.** We describe from the clean cohort, not the raw registry, so the descriptive numbers match exactly the patients and the variable definitions that the models use. No imputation; missingness is reported for every variable. The odds ratios are computed without statsmodels, since that package is not installed in MyDRE, so the screen also populates for the continuous predictors.

## 1. Setup and load

In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
from scipy import stats as st
from sklearn.linear_model import LogisticRegression
THESIS=Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DERIVED=THESIS/"data_derived"; OUT=THESIS/"outputs"/"descriptives"; OUT.mkdir(parents=True, exist_ok=True)
df=pd.read_csv(DERIVED/"cohort_clean.csv"); groups=pd.read_csv(DERIVED/"cohort_columns.csv")
grp=dict(zip(groups['column'],groups['group'])); N=len(df)
print('cohort', df.shape)
print(groups['group'].value_counts().to_string())

## 2. Helper functions

In [ ]:
def desc_cont(cols):
    rows=[]
    for c in cols:
        if c not in df.columns: continue
        s=pd.to_numeric(df[c],errors='coerce'); nn=int(s.notna().sum())
        rows.append({'variable':c,'n':nn,'missing_pct':round(100*(N-nn)/N,1),'mean':round(s.mean(),2),'sd':round(s.std(),2),
                     'median':round(s.median(),2),'q1':round(s.quantile(.25),2),'q3':round(s.quantile(.75),2),
                     'min':round(s.min(),2),'max':round(s.max(),2)})
    return pd.DataFrame(rows)
def desc_levels(cols):
    rows=[]
    for c in cols:
        if c not in df.columns: continue
        for lv,ct in df[c].value_counts(dropna=False).items():
            rows.append({'variable':c,'level':'missing' if pd.isna(lv) else lv,'n':int(ct),'pct_of_cohort':round(100*ct/N,1)})
    return pd.DataFrame(rows)
def desc_bin(cols, eq1=True):
    rows=[]
    for c in cols:
        if c not in df.columns: continue
        s=pd.to_numeric(df[c],errors='coerce'); nn=int(s.notna().sum()); pos=int((s==1).sum()) if eq1 else int((s>=1).sum())
        rows.append({'variable':c,'n_known':nn,'missing_pct':round(100*(N-nn)/N,1),'n_present':pos,'prevalence_pct':round(100*pos/nn,1) if nn else np.nan})
    return pd.DataFrame(rows)
def unadj_or(y,x):
    d=pd.DataFrame({'y':pd.to_numeric(y,errors='coerce'),'x':pd.to_numeric(x,errors='coerce')}).dropna()
    out={'OR':np.nan,'lo':np.nan,'hi':np.nan,'p':np.nan,'n':len(d)}
    if len(d)<30 or d['y'].nunique()<2 or d['x'].nunique()<2: return out
    try:
        m=LogisticRegression(penalty='l2',C=1e8,solver='lbfgs',max_iter=3000).fit(d[['x']].values,d['y'].values)
        b=float(m.coef_[0,0]); p=m.predict_proba(d[['x']].values)[:,1]; W=p*(1-p)
        X=np.column_stack([np.ones(len(d)),d['x'].values]); se=float(np.sqrt(np.diag(np.linalg.inv(X.T@(X*W[:,None])))[1]))
        out.update(OR=round(np.exp(b),2),lo=round(np.exp(b-1.96*se),2),hi=round(np.exp(b+1.96*se),2),p=float(2*(1-st.norm.cdf(abs(b/se)))))
    except Exception: pass
    return out
def fmt_p(p): return '' if (p is None or p!=p) else ('<0.001' if p<0.001 else round(p,3))
print('helpers ready')

## 3. Marginal descriptives by predictor group

In [ ]:
CONT=['age_at_surgery','bmi','charlson_cci','weight_loss_kg','ct_stage','cn_stage','op_duration_min','blood_loss_ml','icu_days','los_days']
BIN=['sex_male','comlong','group_cardiovascular','anamok_prior_surgery','immunosuppression','dietitian_preop',
     'neoadj_received','neoadj_chemoradiation','neoadj_completed','smoking_current','smoking_ever','histology_adeno',
     'histology_squamous','surgical_approach_mis','resection_transthoracic','anastomosis_cervical','conversion','intraop_complication']
LEV=['asascore','comdiam_ord','tumlok1','histology','resection_type','anastomosis_location','reconstruction_type']
predictors_continuous=desc_cont(CONT)
predictors_binary=desc_bin(BIN).sort_values('prevalence_pct',ascending=False)
predictors_levels=desc_levels(LEV)
print(predictors_continuous.to_string(index=False)); print(); print(predictors_binary.to_string(index=False))

## 4. Outcome prevalences

In [ ]:
OUT_BIN=['any_complication','cd_ge_II','cd_ge_IIIa','cd_ge_IIIb','cd_ge_IV','mortality_cd_V','mortality_30d','mortality_90d',
         'pulmonary','readmission','reintervention','los_prolonged','discharge_home','textbook_outcome']
outcomes=desc_bin(OUT_BIN)
los_summary=desc_cont(['los_days','icu_days'])
print(outcomes.to_string(index=False))

## 5. Table 1 by the two headline outcomes

**Decision point.** We show the predictors split by the two headline outcomes, major complications (Clavien IIIb or higher) and prolonged stay, with a p value for the difference, Mann Whitney for continuous and chi square for categorical. This is the descriptive backbone of variable selection: it shows which factors differ between patients who do and do not experience the outcome.

In [ ]:
def table1(outcome):
    y=pd.to_numeric(df[outcome],errors='coerce'); m=y.notna(); y2=y[m].astype(int); sub=df[m].copy()
    n0=int((y2==0).sum()); n1=int((y2==1).sum())
    sub['asa_ge3']=(pd.to_numeric(sub['asascore'],errors='coerce')>=3).astype(float)
    def mi(s): s=pd.to_numeric(s,errors='coerce').dropna(); return f"{s.median():.1f} [{s.quantile(.25):.1f}, {s.quantile(.75):.1f}]" if len(s) else ''
    def mwu(x):
        d=pd.DataFrame({'x':pd.to_numeric(x,errors='coerce'),'y':y2.values}).dropna(); g1=d.loc[d.y==1,'x']; g0=d.loc[d.y==0,'x']
        return st.mannwhitneyu(g1,g0).pvalue if (len(g1)>=3 and len(g0)>=3) else np.nan
    def chi(x):
        d=pd.DataFrame({'x':pd.to_numeric(x,errors='coerce'),'y':y2.values}).dropna(); ct=pd.crosstab(d.x,d.y)
        return st.chi2_contingency(ct)[1] if (ct.shape[0]>=2 and ct.shape[1]>=2) else np.nan
    rows=[]
    for c in ['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage','op_duration_min','blood_loss_ml']:
        rows.append({'Variable':c+' med [IQR]','Overall':mi(sub[c]),f'No (n={n0})':mi(sub.loc[y2.values==0,c]),f'Yes (n={n1})':mi(sub.loc[y2.values==1,c]),'p':fmt_p(mwu(sub[c]))})
    for c,lab in [('sex_male','male'),('asa_ge3','ASA 3+'),('comlong','COPD'),('neoadj_received','neoadjuvant'),
                  ('histology_adeno','adenocarcinoma'),('surgical_approach_mis','minimally invasive'),('anastomosis_cervical','cervical anastomosis')]:
        s=pd.to_numeric(sub[c],errors='coerce')
        def npct(mask): ss=s[mask].dropna(); return f"{int((ss==1).sum())} ({100*(ss==1).mean():.1f}%)" if len(ss) else ''
        rows.append({'Variable':lab+' n (%)','Overall':npct(s.index==s.index),f'No (n={n0})':npct(y2.values==0),f'Yes (n={n1})':npct(y2.values==1),'p':fmt_p(chi(s))})
    return pd.DataFrame(rows)
table1_major=table1('cd_ge_IIIb'); table1_prolonged=table1('los_prolonged')
print('=== Table 1 by major complications ==='); print(table1_major.to_string(index=False))

## 6. Univariable odds ratio screen across outcomes

**Insight to carry forward.** This screen shows the unadjusted association of each predictor with each outcome. It is the evidence behind which predictors the models lean on, and it is unadjusted, not a multivariable model.

In [ ]:
SCREEN_PRED=['age_at_surgery','bmi','asascore','comlong','charlson_cci','comdiam_ord','neoadj_received',
             'ct_stage','cn_stage','histology_adeno','surgical_approach_mis','anastomosis_cervical','op_duration_min','blood_loss_ml']
SCREEN_OUT=['pulmonary','cd_ge_IIIb','reintervention','los_prolonged','textbook_outcome']
recs=[]
for o in SCREEN_OUT:
    for p in SCREEN_PRED:
        if p in df.columns:
            r=unadj_or(df[o],df[p]); r.update(outcome=o,predictor=p); recs.append(r)
u=pd.DataFrame(recs)
u['cell']=u.apply(lambda r:'' if r['OR']!=r['OR'] else f"{r['OR']} ({'<0.001' if (r['p']==r['p'] and r['p']<0.001) else (round(r['p'],3) if r['p']==r['p'] else 'na')})",axis=1)
univariable_matrix=u.pivot(index='predictor',columns='outcome',values='cell').reindex([p for p in SCREEN_PRED if p in df.columns])[SCREEN_OUT]
univariable_long=u[['outcome','predictor','n','OR','lo','hi','p']].copy()
univariable_long['p']=univariable_long['p'].apply(lambda v:'' if v!=v else ('<0.001' if v<0.001 else round(v,4)))
print(univariable_matrix.to_string())

## 7. Missingness ranking

In [ ]:
mrows=[{'variable':c,'missing_pct':round(100*df[c].isna().mean(),1),'n_known':int(df[c].notna().sum()),'group':grp.get(c,'')} for c in df.columns]
missingness_ranking=pd.DataFrame(mrows).sort_values('missing_pct',ascending=False)
print(missingness_ranking.head(20).to_string(index=False))

## 8. Information sheet, key insights, and the single workbook

In [ ]:
info=pd.DataFrame([
 ('Purpose','Descriptive statistics and prevalence for the upper GI cohort, read from cohort_clean.csv (single source of truth).'),
 ('Cohort','n={}. Oesophageal {}, gastric {}, unclassified {}.'.format(N,int((df.cancer_subtype=="oesophageal").sum()),int((df.cancer_subtype=="gastric").sum()),int((df.cancer_subtype=="unclassified").sum()))),
 ('Missing data','Complete case, no imputation. Missingness reported per variable.'),
 ('Odds ratios','Univariable, unadjusted, computed without statsmodels so they populate for continuous predictors too.'),
 ('Headline outcomes','Major complications (Clavien IIIb or higher) and prolonged stay. All outcomes are described.'),
 ('Charlson','charlson_cci is the composite comorbidity score built in preprocessing from the registry components.'),
 ('Sheets','Predictors continuous and categorical, outcomes, Table 1 by each headline outcome, the univariable screen, and the missingness ranking.'),
], columns=['Item','Detail'])
insights=pd.DataFrame([
 ('ASA and neoadjuvant vary little','Most patients are ASA 2 to 3 and 94% received neoadjuvant therapy, which limits how much either can discriminate.'),
 ('Comorbidity is sparse','Captured better by the Charlson score and COPD than by individual conditions.'),
 ('Clinical stage adds spread','cN spreads across N0 to N3 and is a genuine new predictor; cT is concentrated at T3.'),
 ('Outcome balance','Any complication 57%, Clavien II 47%, major IIIb 15%, pulmonary 24%, prolonged stay 24%, reintervention 23%, Textbook Outcome 59%. Mortality 2 to 3% is too rare to model alone.'),
], columns=['Insight','Detail'])
sheets=[('Information',info),('Key_insights',insights),('Predictors_continuous',predictors_continuous),
        ('Predictors_binary',predictors_binary),('Predictors_levels',predictors_levels),('Outcomes',outcomes),
        ('LOS_ICU_summary',los_summary),('Table1_major',table1_major),('Table1_prolonged',table1_prolonged),
        ('Univariable_screen',univariable_matrix.reset_index()),('Univariable_long',univariable_long),
        ('Missingness_ranking',missingness_ranking)]
xlsx=OUT/'duca_descriptives_for_review.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    for name,d in sheets: (d if len(d) else pd.DataFrame({'note':['empty']})).to_excel(xw,sheet_name=name[:31],index=False)
    from openpyxl.styles import Font, Alignment
    for name,d in sheets:
        ws=xw.sheets[name[:31]]; ws.freeze_panes='A2'
        for j,col in enumerate((list(d.columns) if len(d) else ['note']),start=1):
            ws.cell(row=1,column=j).font=Font(bold=True)
            try: w=max([len(str(col))]+[len(str(v)) for v in d[col].astype(str).values[:200]])
            except Exception: w=14
            ws.column_dimensions[ws.cell(row=1,column=j).column_letter].width=min(max(12,w+2),60)
        if name in ('Information','Key_insights'):
            ws.column_dimensions['B'].width=95
            for r in range(1,len(d)+2): ws.cell(row=r,column=2).alignment=Alignment(wrap_text=True,vertical='top')
print('wrote', xlsx)

## 9. Notes

* This workbook is the descriptive companion to the modelling. It reads only cohort_clean.csv, so the numbers match the models exactly.
* The univariable screen and the two Table 1 sheets are the variable selection evidence. The screen is unadjusted, so it shows association strength and direction, not an adjusted effect.
* Complication type breakdowns are not here, because cohort_clean keeps only the derived outcomes; if Marieke wants the type breakdown we can add it from the raw registry separately.